# Advanced Phishing Detection - Data Exploration

This notebook explores the collected phishing and legitimate URL datasets, analyzes patterns, and prepares data for advanced deep learning models.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# Set style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

# Import our modules
import sys
sys.path.append('../src')
from data_collection.data_collector import PhishingDataCollector
from preprocessing.feature_engineering import AdvancedFeatureExtractor, create_feature_pipeline

## 1. Data Collection and Loading

In [ ]:
# Initialize data collector
collector = PhishingDataCollector(data_dir="../data")

# Collect data from all sources
print("Collecting phishing and legitimate URLs...")
collector.collect_all_data(phishing_limit=5000, legitimate_limit=5000)

# Get balanced dataset
train_df, test_df = collector.get_dataset(balance_classes=True)

print(f"Training set: {len(train_df)} URLs")
print(f"Test set: {len(test_df)} URLs")
print(f"Class distribution in training set:")
print(train_df['label'].value_counts())

## 2. Basic Dataset Statistics

In [ ]:
# Combine datasets for analysis
full_df = pd.concat([train_df, test_df], ignore_index=True)

print("Dataset Overview:")
print(f"Total URLs: {len(full_df)}")
print(f"Phishing URLs: {len(full_df[full_df['label'] == 1])} ({len(full_df[full_df['label'] == 1])/len(full_df)*100:.1f}%)")
print(f"Legitimate URLs: {len(full_df[full_df['label'] == 0])} ({len(full_df[full_df['label'] == 0])/len(full_df)*100:.1f}%)")
print(f"\nData Sources:")
print(full_df['source'].value_counts())

In [ ]:
# Visualize class distribution
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Class distribution
full_df['label'].value_counts().plot(kind='bar', ax=axes[0])
axes[0].set_title('Class Distribution')
axes[0].set_xlabel('Label (0: Legitimate, 1: Phishing)')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=0)

# Source distribution
full_df['source'].value_counts().plot(kind='bar', ax=axes[1])
axes[1].set_title('Data Source Distribution')
axes[1].set_xlabel('Source')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 3. URL Length Analysis

In [ ]:
# Calculate URL lengths
full_df['url_length'] = full_df['url'].str.len()

# Basic statistics
print("URL Length Statistics:")
print(full_df.groupby('label')['url_length'].describe())

In [ ]:
# Visualize URL length distribution
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Histogram
phishing_lengths = full_df[full_df['label'] == 1]['url_length']
legitimate_lengths = full_df[full_df['label'] == 0]['url_length']

axes[0, 0].hist(legitimate_lengths, bins=50, alpha=0.7, label='Legitimate', density=True)
axes[0, 0].hist(phishing_lengths, bins=50, alpha=0.7, label='Phishing', density=True)
axes[0, 0].set_title('URL Length Distribution')
axes[0, 0].set_xlabel('URL Length')
axes[0, 0].set_ylabel('Density')
axes[0, 0].legend()

# Box plot
full_df.boxplot(column='url_length', by='label', ax=axes[0, 1])
axes[0, 1].set_title('URL Length by Class')
axes[0, 1].set_xlabel('Label (0: Legitimate, 1: Phishing)')
axes[0, 1].set_ylabel('URL Length')

# Log scale histogram
axes[1, 0].hist(legitimate_lengths, bins=50, alpha=0.7, label='Legitimate', density=True)
axes[1, 0].hist(phishing_lengths, bins=50, alpha=0.7, label='Phishing', density=True)
axes[1, 0].set_yscale('log')
axes[1, 0].set_title('URL Length Distribution (Log Scale)')
axes[1, 0].set_xlabel('URL Length')
axes[1, 0].set_ylabel('Log Density')
axes[1, 0].legend()

# Cumulative distribution
axes[1, 1].hist(legitimate_lengths, bins=100, alpha=0.7, label='Legitimate', 
               density=True, cumulative=True, histtype='step', linewidth=2)
axes[1, 1].hist(phishing_lengths, bins=100, alpha=0.7, label='Phishing', 
               density=True, cumulative=True, histtype='step', linewidth=2)
axes[1, 1].set_title('Cumulative URL Length Distribution')
axes[1, 1].set_xlabel('URL Length')
axes[1, 1].set_ylabel('Cumulative Probability')
axes[1, 1].legend()

plt.tight_layout()
plt.show()

## 4. Advanced Feature Extraction and Analysis

In [ ]:
# Extract features for a sample of URLs (to avoid long processing time)
sample_size = 1000
sample_df = full_df.sample(n=sample_size, random_state=42)

print(f"Extracting features for {sample_size} URLs...")
features_df = create_feature_pipeline(sample_df['url'].tolist(), max_workers=5)

# Add labels
features_df['label'] = sample_df['label'].values

print(f"Extracted {len(features_df.columns)-1} features")
print(f"Feature extraction completed for {len(features_df)} URLs")

In [ ]:
# Display feature statistics
numeric_features = features_df.select_dtypes(include=[np.number]).columns.tolist()
numeric_features.remove('label')

print("Top 20 Most Discriminative Features:")
feature_importance = []

for feature in numeric_features:
    phishing_mean = features_df[features_df['label'] == 1][feature].mean()
    legitimate_mean = features_df[features_df['label'] == 0][feature].mean()
    
    if legitimate_mean != 0:
        ratio = abs(phishing_mean - legitimate_mean) / abs(legitimate_mean)
    else:
        ratio = abs(phishing_mean - legitimate_mean)
    
    feature_importance.append((feature, ratio, phishing_mean, legitimate_mean))

feature_importance.sort(key=lambda x: x[1], reverse=True)

for i, (feature, importance, phish_mean, legit_mean) in enumerate(feature_importance[:20]):
    print(f"{i+1:2d}. {feature:25s} | Ratio: {importance:.3f} | Phishing: {phish_mean:.3f} | Legitimate: {legit_mean:.3f}")

In [ ]:
# Visualize top discriminative features
top_features = [f[0] for f in feature_importance[:12]]

fig, axes = plt.subplots(3, 4, figsize=(20, 15))
axes = axes.ravel()

for i, feature in enumerate(top_features):
    phishing_data = features_df[features_df['label'] == 1][feature]
    legitimate_data = features_df[features_df['label'] == 0][feature]
    
    axes[i].hist(legitimate_data, bins=30, alpha=0.7, label='Legitimate', density=True)
    axes[i].hist(phishing_data, bins=30, alpha=0.7, label='Phishing', density=True)
    axes[i].set_title(f'{feature}')
    axes[i].set_ylabel('Density')
    axes[i].legend()
    axes[i].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Character-Level Analysis for URLNet

In [ ]:
# Character frequency analysis
def analyze_character_patterns(urls, label_name):
    all_chars = ''.join(urls)
    char_freq = {}
    
    for char in all_chars:
        char_freq[char] = char_freq.get(char, 0) + 1
    
    # Sort by frequency
    sorted_chars = sorted(char_freq.items(), key=lambda x: x[1], reverse=True)
    
    print(f"\n{label_name} URLs - Top 20 Characters:")
    for i, (char, freq) in enumerate(sorted_chars[:20]):
        char_display = repr(char) if char in '\n\t\r' else char
        print(f"{i+1:2d}. '{char_display}': {freq:,} ({freq/len(all_chars)*100:.2f}%)")
    
    return char_freq

# Analyze character patterns
phishing_urls = sample_df[sample_df['label'] == 1]['url'].tolist()
legitimate_urls = sample_df[sample_df['label'] == 0]['url'].tolist()

phishing_chars = analyze_character_patterns(phishing_urls, "Phishing")
legitimate_chars = analyze_character_patterns(legitimate_urls, "Legitimate")

In [ ]:
# Character distribution comparison
common_chars = set(phishing_chars.keys()) & set(legitimate_chars.keys())
char_comparison = []

for char in common_chars:
    phish_freq = phishing_chars[char] / sum(phishing_chars.values())
    legit_freq = legitimate_chars[char] / sum(legitimate_chars.values())
    ratio = phish_freq / legit_freq if legit_freq > 0 else float('inf')
    char_comparison.append((char, ratio, phish_freq, legit_freq))

char_comparison.sort(key=lambda x: abs(x[1] - 1), reverse=True)

print("\nMost Discriminative Characters (by frequency ratio):")
for i, (char, ratio, phish_freq, legit_freq) in enumerate(char_comparison[:15]):
    char_display = repr(char) if char in '\n\t\r' else char
    print(f"{i+1:2d}. '{char_display}': Ratio {ratio:.3f} | Phishing: {phish_freq:.4f} | Legitimate: {legit_freq:.4f}")

## 6. Domain Analysis

In [ ]:
# Extract domains from URLs
from urllib.parse import urlparse
import tldextract

def extract_domain_info(url):
    try:
        parsed = urlparse(url)
        extracted = tldextract.extract(url)
        
        return {
            'domain': extracted.domain,
            'subdomain': extracted.subdomain,
            'suffix': extracted.suffix,
            'full_domain': f"{extracted.domain}.{extracted.suffix}" if extracted.suffix else extracted.domain
        }
    except:
        return {'domain': '', 'subdomain': '', 'suffix': '', 'full_domain': ''}

# Extract domain information
domain_info = sample_df['url'].apply(extract_domain_info)
domain_df = pd.DataFrame(domain_info.tolist())
domain_df['label'] = sample_df['label'].values

print("Domain Analysis:")
print(f"Unique domains: {domain_df['full_domain'].nunique()}")
print(f"Unique TLDs: {domain_df['suffix'].nunique()}")
print(f"URLs with subdomains: {len(domain_df[domain_df['subdomain'] != ''])} ({len(domain_df[domain_df['subdomain'] != ''])/len(domain_df)*100:.1f}%)")

In [ ]:
# TLD analysis
tld_analysis = domain_df.groupby(['suffix', 'label']).size().unstack(fill_value=0)
tld_analysis['total'] = tld_analysis.sum(axis=1)
tld_analysis['phishing_ratio'] = tld_analysis[1] / tld_analysis['total']
tld_analysis = tld_analysis.sort_values('total', ascending=False)

print("\nTop 15 TLDs by frequency:")
print(tld_analysis.head(15)[['total', 'phishing_ratio']].round(3))

In [ ]:
# Visualize TLD distribution
top_tlds = tld_analysis.head(10)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# TLD frequency
top_tlds['total'].plot(kind='bar', ax=axes[0])
axes[0].set_title('Top 10 TLDs by Frequency')
axes[0].set_xlabel('TLD')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=45)

# Phishing ratio
top_tlds['phishing_ratio'].plot(kind='bar', ax=axes[1], color='red', alpha=0.7)
axes[1].set_title('Phishing Ratio by TLD')
axes[1].set_xlabel('TLD')
axes[1].set_ylabel('Phishing Ratio')
axes[1].tick_params(axis='x', rotation=45)
axes[1].axhline(y=0.5, color='black', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

## 7. Feature Correlation Analysis

In [ ]:
# Calculate correlation matrix for top features
top_20_features = [f[0] for f in feature_importance[:20]]
correlation_matrix = features_df[top_20_features + ['label']].corr()

# Visualize correlation matrix
plt.figure(figsize=(12, 10))
mask = np.triu(np.ones_like(correlation_matrix, dtype=bool))
sns.heatmap(correlation_matrix, mask=mask, annot=True, cmap='coolwarm', center=0,
            square=True, linewidths=0.5, cbar_kws={"shrink": .8})
plt.title('Feature Correlation Matrix (Top 20 Features)')
plt.tight_layout()
plt.show()

In [ ]:
# Feature correlation with target
target_correlation = correlation_matrix['label'].abs().sort_values(ascending=False)
target_correlation = target_correlation.drop('label')

print("Features most correlated with phishing label:")
for i, (feature, corr) in enumerate(target_correlation.head(15).items()):
    print(f"{i+1:2d}. {feature:25s}: {corr:.4f}")

# Visualize
plt.figure(figsize=(10, 8))
target_correlation.head(15).plot(kind='barh')
plt.title('Top 15 Features by Correlation with Phishing Label')
plt.xlabel('Absolute Correlation')
plt.tight_layout()
plt.show()

## 8. Data Quality Assessment

In [ ]:
# Check for missing values
missing_values = features_df.isnull().sum()
missing_percentage = (missing_values / len(features_df)) * 100

missing_df = pd.DataFrame({
    'Missing Count': missing_values,
    'Missing Percentage': missing_percentage
})
missing_df = missing_df[missing_df['Missing Count'] > 0].sort_values('Missing Count', ascending=False)

if len(missing_df) > 0:
    print("Features with missing values:")
    print(missing_df)
else:
    print("No missing values found in the dataset!")

In [ ]:
# Check for outliers using IQR method
def detect_outliers(df, feature):
    Q1 = df[feature].quantile(0.25)
    Q3 = df[feature].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    outliers = df[(df[feature] < lower_bound) | (df[feature] > upper_bound)]
    return len(outliers)

outlier_counts = {}
for feature in numeric_features[:10]:  # Check top 10 features
    outlier_count = detect_outliers(features_df, feature)
    outlier_counts[feature] = outlier_count

print("Outlier counts for top 10 features:")
for feature, count in outlier_counts.items():
    percentage = (count / len(features_df)) * 100
    print(f"{feature:25s}: {count:4d} ({percentage:.1f}%)")

## 9. Summary and Insights

In [ ]:
print("=" * 60)
print("DATA EXPLORATION SUMMARY")
print("=" * 60)

print(f"\n📊 Dataset Statistics:")
print(f"   • Total URLs analyzed: {len(full_df):,}")
print(f"   • Phishing URLs: {len(full_df[full_df['label'] == 1]):,}")
print(f"   • Legitimate URLs: {len(full_df[full_df['label'] == 0]):,}")
print(f"   • Features extracted: {len(numeric_features)}")

print(f"\n🔍 Key Findings:")
print(f"   • Average phishing URL length: {full_df[full_df['label'] == 1]['url_length'].mean():.1f} chars")
print(f"   • Average legitimate URL length: {full_df[full_df['label'] == 0]['url_length'].mean():.1f} chars")
print(f"   • Most discriminative feature: {feature_importance[0][0]}")
print(f"   • Highest correlation with phishing: {target_correlation.index[0]} ({target_correlation.iloc[0]:.3f})")

print(f"\n🎯 Model Readiness:")
print(f"   • Data quality: {'✅ Excellent' if len(missing_df) == 0 else '⚠️ Needs attention'}")
print(f"   • Class balance: {'✅ Balanced' if abs(len(full_df[full_df['label'] == 1]) - len(full_df[full_df['label'] == 0])) < 100 else '⚠️ Imbalanced'}")
print(f"   • Feature diversity: ✅ {len(numeric_features)} numerical features")

print(f"\n🚀 Next Steps:")
print(f"   1. Train URLNet model with character-level features")
print(f"   2. Train Transformer model with tokenized URLs")
print(f"   3. Implement ensemble methods")
print(f"   4. Deploy real-time API")

print("\n" + "=" * 60)

In [ ]:
# Save processed data for model training
train_df.to_csv('../data/processed/train_urls.csv', index=False)
test_df.to_csv('../data/processed/test_urls.csv', index=False)
features_df.to_csv('../data/processed/sample_features.csv', index=False)

print("✅ Processed data saved to ../data/processed/")
print("   • train_urls.csv: Training URLs")
print("   • test_urls.csv: Test URLs")
print("   • sample_features.csv: Extracted features sample")